In [ ]:
import os
import sys
import json
import torch
import glob
import subprocess
from pathlib import Path

MODEL = 'gru'

DATASET_PATH = Path("/kaggle/input/skripsi-edgenmten-id")
TOKENIZER_MODEL = Path(glob.glob("/kaggle/input/**/spm_en_id.model", recursive=True)[0])

REPO_URL = 'https://github.com/0wLzz/Edge-NMT.git'

if not os.path.isdir('Edge-NMT'):
    subprocess.run(['git','clone','--depth','1',REPO_URL], check=True)

os.chdir('/kaggle/working/Edge-NMT')
sys.path.insert(0, os.getcwd())
print('Current Working Directory:', os.getcwd())

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','datasets<3.0','sentencepiece','sacrebleu','pyyaml','optuna','torchinfo'], check=True)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print('GPU:', torch.cuda.get_device_name(0), 'sm_%d%d' % cap)
    assert cap[0] >= 7, 'Unsupported GPU (need T4 sm_75); set Accelerator=GPU T4 x2.'
else:
    print('WARNING: CPU only')

In [ ]:
CFG = "configs/config.yaml"

def run_module(mod, *args):
    print("RUN", mod, *args, flush=True)
    subprocess.run([sys.executable, "-m", mod, *map(str, args)], check=True)

run_module(
    "model.training.hyperparameter_search",
    "--arch", MODEL,
    "--config", CFG,
    "--dataset-dir", DATASET_PATH,
    "--tokenizer-model", TOKENIZER_PATH / "spm_en_id.model",
    "--n-trials", "30",
)

In [ ]:
p = 'results/hparam_search/best_gru.json'

if os.path.exists(p):
    b = json.load(open(p))

    print('BEST GRU val_loss=%.4f' % b['val_loss'])
    print('hparams:', b['hparams'])
    print('total_params:', b.get('total_params'))
    print('breakdown:', b.get('param_breakdown'))

else: 
    print('no best_gru.json — see search output above')